# บทที่ 2: อาร์เรย์, Python List และ Recursion

> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)
> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*
> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)

---

## 2.1 Array vs Python List

- **Array** ในความหมายคลาสสิก = บล็อกหน่วยความจำต่อเนื่อง ขนาดคงที่ เข้าถึงด้วย index ได้ \(O(1)\)
- **Python list** = *dynamic array* เก็บ pointer ไปยัง object เพิ่มขนาดอัตโนมัติด้วยการ **resize** (จองใหม่แล้วคัดลอก)
- `append` มีต้นทุน **amortized \(O(1)\)** เพราะการ resize เกิดไม่บ่อย (โตแบบเท่าตัวโดยประมาณ)

| การดำเนินการ | Complexity | หมายเหตุ |
|---|---|---|
| `a[i]` | \(O(1)\) | random access |
| `a.append(x)` | \(O(1)\) amortized | อาจ resize |
| `a.insert(0, x)` | \(O(n)\) | ต้องเลื่อนสมาชิก |
| `a.pop()` | \(O(1)\) | ท้ายลิสต์ |
| `a.pop(0)` | \(O(n)\) | หัวลิสต์ |
| `x in a` | \(O(n)\) | linear search |

In [1]:
import sys

# สังเกตการเติบโตของหน่วยความจำที่ Python จองไว้ให้ list (over-allocation)
a = []
prev = sys.getsizeof(a)
print(f"len=0  bytes={prev}")
for i in range(1, 33):
    a.append(i)
    cur = sys.getsizeof(a)
    if cur != prev:                     # พิมพ์เฉพาะจุดที่เกิดการ resize
        print(f"len={len(a):>3}  bytes={cur}  <-- resize")
        prev = cur

len=0  bytes=56
len=  1  bytes=88  <-- resize
len=  5  bytes=120  <-- resize
len=  9  bytes=184  <-- resize
len= 17  bytes=248  <-- resize
len= 25  bytes=312  <-- resize


In [ ]:
# สร้าง Dynamic Array ด้วยตัวเอง เพื่อเข้าใจกลไก resize
import ctypes

class DynamicArray:
    def __init__(self):
        self._n = 0            # จำนวนสมาชิกจริง
        self._cap = 1          # ความจุที่จองไว้
        self._A = self._make_array(self._cap)

    def _make_array(self, cap):
        return (cap * ctypes.py_object)()      # จำลอง raw array แบบภาษา C

    def __len__(self):
        return self._n

    def __getitem__(self, k):
        if not 0 <= k < self._n:
            raise IndexError('index out of range')
        return self._A[k]                       # O(1)

    def append(self, obj):
        if self._n == self._cap:
            self._resize(2 * self._cap)         # โตเป็นเท่าตัว -> amortized O(1)
        self._A[self._n] = obj
        self._n += 1

    def _resize(self, cap):
        B = self._make_array(cap)
        for i in range(self._n):
            B[i] = self._A[i]                   # คัดลอกทั้งหมด O(n)
        self._A, self._cap = B, cap

    def __repr__(self):
        return "[" + ", ".join(repr(self._A[i]) for i in range(self._n)) + "]"

da = DynamicArray()
for x in range(6):
    da.append(x * x)
print(da, "len =", len(da), "cap =", da._cap, "da[3] =", da[3])

## 2.2 เทคนิคพื้นฐานกับอาร์เรย์

1. **Two pointers** – ใช้ตัวชี้สองตัววิ่งเข้าหากัน/ตามกัน ลด \(O(n^2)\) ให้เป็น \(O(n)\)
2. **Sliding window** – หน้าต่างเลื่อนสำหรับปัญหาช่วงต่อเนื่อง
3. **Prefix sum** – เตรียมผลรวมสะสม ตอบคำถามช่วง \(O(1)\)
4. **Hashing** – แลกหน่วยความจำเพื่อความเร็ว

In [ ]:
# 1) Two pointers: หาคู่ที่ผลรวม = target ในอาร์เรย์ "ที่เรียงแล้ว"  -> O(n)
def two_sum_sorted(a, target):
    i, j = 0, len(a) - 1
    while i < j:
        s = a[i] + a[j]
        if s == target:
            return (i, j)
        elif s < target:
            i += 1          # ต้องการผลรวมมากขึ้น
        else:
            j -= 1          # ต้องการผลรวมน้อยลง
    return None

# 2) Sliding window: ผลรวมสูงสุดของสมาชิกติดกัน k ตัว -> O(n)
def max_sum_window(a, k):
    window = sum(a[:k])
    best = window
    for i in range(k, len(a)):
        window += a[i] - a[i - k]     # เพิ่มตัวใหม่ ลบตัวที่หลุดหน้าต่าง
        best = max(best, window)
    return best

# 3) Prefix sum: ตอบผลรวมช่วง [l, r] ได้ O(1) ต่อคำถาม
class PrefixSum:
    def __init__(self, a):
        self.p = [0]
        for x in a:
            self.p.append(self.p[-1] + x)
    def range_sum(self, l, r):
        return self.p[r + 1] - self.p[l]

print(two_sum_sorted([1, 3, 4, 7, 11], 11))
print(max_sum_window([2, 1, 5, 1, 3, 2], 3))
ps = PrefixSum([2, 1, 5, 1, 3, 2])
print(ps.range_sum(1, 3))

In [ ]:
# Kadane's algorithm: ผลรวมช่วงต่อเนื่องที่มากที่สุด (maximum subarray) -> O(n)
def max_subarray(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)   # เริ่มใหม่ที่ x หรือรวมกับช่วงเดิม
        best = max(best, cur)
    return best

print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]))   # -> 6 จาก [4,-1,2,1]

## 2.3 Recursion

องค์ประกอบที่ต้องมีทุกครั้ง
1. **Base case** – เงื่อนไขหยุด (ถ้าไม่มี → `RecursionError`)
2. **Recursive case** – ลดขนาดปัญหาแล้วเรียกตัวเอง

Recursion ใช้ **call stack** → space complexity อย่างน้อยเท่ากับความลึกของการเรียก

In [ ]:
import sys
print("recursion limit ปัจจุบัน:", sys.getrecursionlimit())

def factorial(n):
    if n <= 1:                  # base case
        return 1
    return n * factorial(n - 1) # recursive case, ความลึก O(n)

def hanoi(n, src='A', dst='C', aux='B', moves=None):
    # Tower of Hanoi: ต้องย้าย 2^n - 1 ครั้ง
    if moves is None:
        moves = []
    if n == 0:
        return moves
    hanoi(n - 1, src, aux, dst, moves)
    moves.append((src, dst))
    hanoi(n - 1, aux, dst, src, moves)
    return moves

def subsets(a):
    # สร้างสับเซตทั้งหมด (power set) -> 2^n ชุด
    if not a:
        return [[]]
    rest = subsets(a[1:])
    return rest + [[a[0]] + r for r in rest]

print("5! =", factorial(5))
m = hanoi(3)
print("Hanoi n=3 ใช้", len(m), "ครั้ง:", m)
print("subsets([1,2,3]) =", subsets([1, 2, 3]))

---
## แบบฝึกหัดบทที่ 2

1. หมุนอาร์เรย์ไปทางซ้าย k ตำแหน่ง โดยใช้หน่วยความจำเพิ่ม \(O(1)\)
2. หาสมาชิกที่ปรากฏเพียงครั้งเดียว เมื่อสมาชิกอื่นปรากฏ 2 ครั้ง ด้วย \(O(n)\) time / \(O(1)\) space
3. ย้ายเลข 0 ทั้งหมดไปท้ายอาร์เรย์ โดยรักษาลำดับของตัวที่ไม่ใช่ 0
4. เขียน recursion ตรวจว่าสตริงเป็น palindrome หรือไม่

### เฉลยบทที่ 2

In [ ]:
# ข้อ 1: เทคนิค reverse 3 ครั้ง -> O(n) time, O(1) space
def rotate_left(a, k):
    n = len(a)
    k %= n
    def rev(i, j):
        while i < j:
            a[i], a[j] = a[j], a[i]
            i, j = i + 1, j - 1
    rev(0, k - 1); rev(k, n - 1); rev(0, n - 1)
    return a

# ข้อ 2: XOR ทุกตัว ตัวที่ซ้ำกันจะหักล้างกันเอง
def single_number(a):
    r = 0
    for x in a:
        r ^= x
    return r

# ข้อ 3: two pointers เขียนทับตำแหน่ง write
def move_zeros(a):
    w = 0
    for i in range(len(a)):
        if a[i] != 0:
            a[w], a[i] = a[i], a[w]
            w += 1
    return a

# ข้อ 4
def is_palindrome(s, i=0, j=None):
    if j is None:
        j = len(s) - 1
    if i >= j:
        return True
    return s[i] == s[j] and is_palindrome(s, i + 1, j - 1)

print(rotate_left([1,2,3,4,5,6,7], 3))
print(single_number([4,1,2,1,2]))
print(move_zeros([0,1,0,3,12]))
print(is_palindrome("racecar"), is_palindrome("python"))